In [ ]:
import os
import json
import matplotlib.pylab as plt
import numpy as np
from pathlib import Path
import random
from tyssue.draw.plt_draw import draw_edge, draw_vert
import traceback
import sys



from scipy.optimize import curve_fit
from matplotlib.colors import ListedColormap


# Importing my modules

repo_root = Path.cwd()
while not (repo_root / "src").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
sys.path.append(str(repo_root))

import src.vertexModel1 as vertexModel1
import src.vertexModel2 as vertexModel2

import src.inputMechanicalParametersModel1 as MechanicalParams1
import src.inputMechanicalParametersModel2 as MechanicalParams2


import src.auxFunctions as auxFunctions



import warnings
warnings.filterwarnings("ignore")

### Model 1 collagenase laser ablations

### Carrying out laser ablations of different ECM:FRC length elasticity ratios in Model 1 to choose the right split, by comparing results with ex vivo collagenase laser ablation results

In [ ]:
def set_ecm_frc_ratio(cellmap, ecm_ratio, frc_ratio, total_le=90, remove_ecm=False):
    """
    Partitioning length elasticity between ECM and FRC.
    """
    frc_to_ecm_ratio = frc_ratio / ecm_ratio
    frc_le = (frc_to_ecm_ratio / (1 + frc_to_ecm_ratio)) * total_le
    ecm_le = total_le - frc_le
    
    if remove_ecm:
        ecm_le = 0
    
    cellmap.edge_df['length_elasticity_ECM'] = ecm_le
    cellmap.edge_df['length_elasticity_FRC'] = frc_le
    cellmap.edge_df['length_elasticity'] = frc_le + ecm_le
    
    return cellmap

In [ ]:
def initialise_tissue_with_ratio(ratio_ecm_frc, remove_ecm=False, total_le=90):
    """
    Initialising Model 1 with specified ECM:FRC ratio, written as (x,y)
    """
    cellmap, geom, energy_model = vertexModel1.initialize()
    cellmap = MechanicalParams1.update(cellmap)
    
    # Fix outer boundary vertices
    boundary_edges, boundary_faces, inside_edges, outside_edges, inside_faces, inside_vertices, outside_vertices = auxFunctions.identify_boundary_layers(cellmap, 1)
    for vertex_id in outside_vertices:
        if vertex_id in cellmap.vert_df.index:
            cellmap.vert_df.at[vertex_id, "viscosity"] = 1000000
    
    cellmap.update_specs({"vert": {"viscosity": cellmap.vert_df["viscosity"].values}}, reset=True)
    
    ecm_ratio, frc_ratio = ratio_ecm_frc
    cellmap = set_ecm_frc_ratio(cellmap, ecm_ratio, frc_ratio, total_le, remove_ecm)
    
    energy_model.compute_energy(cellmap)
    cellmap, geom, energy_model, _, _ = vertexModel1.solveEuler(
        cellmap, geom, energy_model, endTime=100
    )
    
    return cellmap, geom, energy_model

In [ ]:
def plot_recoil_curve(time_steps, displacement, initial_recoil, k, tissue_id, ablation_id):
    """
    Displaying the recoil displacement curve with fitted model.
    """
    
    fig, ax = plt.subplots(figsize=(8, 6))
    
    # Plotting actual displacement
    ax.plot(time_steps, displacement, 'b-', linewidth=2, label='Actual displacement')
    
    # Plotting fitted model (the same as used ex vivo for analysis)
    fitted_displacement = (initial_recoil / k) * (1 - np.exp(-k * np.array(time_steps)))
    ax.plot(time_steps, fitted_displacement, 'r--', linewidth=2, label=f'Fit: K={k:.3f}, v0={initial_recoil:.5f}')
    
    ax.set_xlabel('Time', fontsize=12)
    ax.set_ylabel('Displacement', fontsize=12)
    ax.set_title(f'Recoil Dynamics - Tissue {tissue_id}, Ablation {ablation_id}', fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.show()
    print(f"Recoil curve displayed")

In [ ]:
def perform_laser_ablation_model1(cellmap, geom, energy_model, endTime=50, ratio_name=None, tissue_id=None, ablation_id=None, plot_dir=None):
    """
    Perform laser ablation on a random interior edge with visualization.
    RETURNS the updated cellmap after ablation.
    """
    # Finding interior edges
    boundary_edges, boundary_faces, inside_edges, outside_edges, inside_faces, inside_vertices, outside_vertices = auxFunctions.identify_boundary_layers(cellmap, max_layers=1)
    if not inside_edges:
        raise ValueError("No inside edges found")
    
    # Choosing random edge and finding its opposite edge
    chosen_edge = random.choice(inside_edges)
    srce = cellmap.edge_df.loc[chosen_edge, "srce"]
    trgt = cellmap.edge_df.loc[chosen_edge, "trgt"]
    
    opposite = cellmap.edge_df[
        (cellmap.edge_df["srce"] == trgt) & (cellmap.edge_df["trgt"] == srce)
    ]
    if opposite.empty:
        raise ValueError("Opposite edge not found")
    opposite_edge = opposite.index[0]
    
    # Visualising edge choice before ablation
    if plot_dir and ratio_name and ablation_id is not None:
        print(f"Tissue before ablation")
        auxFunctions.highlight_edge_on_cellmap(
            cellmap, 
            edge_id=chosen_edge,
            figsize=(12, 12),
            vert_col='orange',
            edge_highlight_col='red',
            save_path=f"{plot_dir}/before_ablation_{ratio_name.replace(':', '_')}_tissue{tissue_id}_abl{ablation_id}.png",
            show_figure=False
        )
    
    # Ablating chosen edge
    cellmap.edge_df.at[chosen_edge, "length_elasticity"] = 0.0
    cellmap.edge_df.at[chosen_edge, "line_tension"] = 0.0
    cellmap.edge_df.at[opposite_edge, "length_elasticity"] = 0.0
    cellmap.edge_df.at[opposite_edge, "line_tension"] = 0.0
    
    # Relaxing the model
    energy_model.compute_energy(cellmap)
    cellmap, geom, energy_model, _, solver = vertexModel1.solveEuler(
        cellmap, geom, energy_model, endTime=endTime
    )
    
    # Visualising after ablation
    if plot_dir and ratio_name and ablation_id is not None:
        print(f"Tissue after ablation")
        auxFunctions.highlight_edge_on_cellmap(
            cellmap, 
            edge_id=chosen_edge,
            figsize=(12, 12),
            vert_col='orange',
            edge_highlight_col='red',
            save_path=f"{plot_dir}/after_ablation_{ratio_name.replace(':', '_')}_tissue{tissue_id}_abl{ablation_id}.png",
            show_figure=False
        )
    
    # Tracking vertex displacement post ablation
    displacement = []
    time_steps = []
    initial_distance = None
    
    for t, cm in solver.history:
        srce_coords = cm.vert_df.loc[srce, ['x', 'y']]
        trgt_coords = cm.vert_df.loc[trgt, ['x', 'y']]
        distance = np.linalg.norm(srce_coords - trgt_coords)
        
        if initial_distance is None:
            initial_distance = distance
        
        displacement.append(distance - initial_distance)
        time_steps.append(t)
    
    # Fitting exponential recoil model
    def recoil_model(t, initial_recoil, k):
        return (initial_recoil / k) * (1 - np.exp(-k * t))
    
    params, _ = curve_fit(recoil_model, time_steps, displacement, 
                          p0=[0.00001, 3], bounds=(0, np.inf))
    initial_recoil, k = params
    
    # Plotting recoil
    if plot_dir and ratio_name and ablation_id is not None:
        plot_recoil_curve(
            time_steps, displacement, initial_recoil, k, 
            tissue_id, ablation_id
        )
    
    return {
        'displacement': displacement,
        'time_steps': time_steps,
        'initial_recoil': float(initial_recoil),
        'k': float(k),
        'chosen_edge': int(chosen_edge),
        'opposite_edge': int(opposite_edge),
        'updated_cellmap': cellmap
    }

In [ ]:
def run_collagenase_experiment(ratio_ecm_frc, 
                                num_tissues=3, 
                                ablations_per_tissue=5,
                                total_le=90,
                                save_path=None,
                                plot_dir=None):
    """
    Running collagenase experiment for a single ECM:FRC ratio with plotting.
    Performs consecutive ablations on the same tissue but re-initialises a new tissue 'num_tissues' times
    """
    ecm_ratio, frc_ratio = ratio_ecm_frc
    ratio_name = f"{ecm_ratio}:{frc_ratio}"
    
    print(f"Testing ratio {ratio_name} (ECM:FRC)")
    print(f"Collagenase: ECM will be removed after initial relaxation")
    
    # Creating plot directory for chosen ratio
    if plot_dir:
        ratio_plot_dir = f"{plot_dir}/ratio_{ecm_ratio}_{frc_ratio}"
        Path(ratio_plot_dir).mkdir(parents=True, exist_ok=True)
    else:
        ratio_plot_dir = None
    
    results = {
        'ratio': ratio_name,
        'ecm_ratio': ecm_ratio,
        'frc_ratio': frc_ratio,
        'total_le': total_le,
        'ablations': []
    }
    
    for tissue_idx in range(num_tissues):
        print(f"Tissue {tissue_idx + 1}/{num_tissues}")
        
        # Initialising with ECM present and allowing tissue to relax to minimum energy state 
        print("Initialising tissue with ECM present")
        cellmap, geom, energy_model = initialise_tissue_with_ratio(
            ratio_ecm_frc, remove_ecm=False, total_le=total_le
        )
        
        # Applying collagenase (removing ECM)
        print("Applying collagenase (removing ECM)")
        cellmap = set_ecm_frc_ratio(cellmap, ecm_ratio, frc_ratio, 
                                     total_le=total_le, remove_ecm=True)
        
        # Relaxing after collagenase
        energy_model.compute_energy(cellmap)
        cellmap, geom, energy_model, _, _ = vertexModel1.solveEuler(
            cellmap, geom, energy_model, endTime=100
        )
        
        # Recording LE values after collagenase (before any ablations)
        frc_le_value = float(cellmap.edge_df['length_elasticity_FRC'].iloc[0])
        ecm_le_value = float(cellmap.edge_df['length_elasticity_ECM'].iloc[0])
        
        # Performing consecutive laser ablations on the same tissue
        for ablation_idx in range(ablations_per_tissue):
            ablation_num = tissue_idx * ablations_per_tissue + ablation_idx + 1
            print(f"Ablation {ablation_idx + 1}/{ablations_per_tissue} (Total: {ablation_num})")
            
            try:
                ablation_result = perform_laser_ablation_model1(
                    cellmap, geom, energy_model, endTime=50,
                    ratio_name=ratio_name, tissue_id=tissue_idx + 1,
                    ablation_id=ablation_num,
                    plot_dir=ratio_plot_dir
                )
                
                cellmap = ablation_result['updated_cellmap']
                del ablation_result['updated_cellmap']
                
                ablation_result['tissue_id'] = tissue_idx + 1
                ablation_result['ablation_id'] = ablation_num
                ablation_result['frc_le'] = frc_le_value
                ablation_result['ecm_le'] = ecm_le_value
                
                results['ablations'].append(ablation_result)
                
                print(f"K = {ablation_result['k']:.3f}, Initial recoil = {ablation_result['initial_recoil']:.5f}")
                
            except Exception as e:
                print(f"Ablation failed: {e}")
                continue
    
    # Summary statistics
    if results['ablations']:
        k_values = [a['k'] for a in results['ablations']]
        recoil_values = [a['initial_recoil'] for a in results['ablations']]
        
        results['summary'] = {
            'num_ablations': len(results['ablations']),
            'mean_k': float(np.mean(k_values)),
            'std_k': float(np.std(k_values)),
            'mean_initial_recoil': float(np.mean(recoil_values)),
            'std_initial_recoil': float(np.std(recoil_values))
        }
    else:
        results['summary'] = {
            'num_ablations': 0,
            'mean_k': None,
            'std_k': None,
            'mean_initial_recoil': None,
            'std_initial_recoil': None
        }

    print(f"\n Complete for ratio {ratio_name}")
    if results['summary']['mean_k']:
        print(f"   Mean K: {results['summary']['mean_k']:.3f} ± {results['summary']['std_k']:.3f}")
        print(f"   Mean Initial Recoil: {results['summary']['mean_initial_recoil']:.5f} ± {results['summary']['std_initial_recoil']:.5f}")
    print(f"   Successful ablations: {len(results['ablations'])}/{num_tissues * ablations_per_tissue}")
    
    # Saving results if path provided
    if save_path:
        with open(save_path, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"   Results saved to: {save_path}")
    
    if ratio_plot_dir:
        print(f"   Plots saved to: {ratio_plot_dir}")
    
    return results

In [ ]:
### Carrying out laser ablations for collagenase experiment on multiple FRC:ECM ratios

def run_multi_ratio_collagenase_experiment(ratios_ecm_frc,
                                            num_tissues=3,
                                            ablations_per_tissue=5,
                                            total_le=90,
                                            output_dir="collagenase_results_Model1",
                                            plot_dir="collagenase_plots_Model1"):
    """
    Run collagenase experiment for multiple ECM:FRC ratios with plotting.
    """
    # Creating output directories
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    if plot_dir:
        Path(plot_dir).mkdir(parents=True, exist_ok=True)
    
    all_results = {}
    
    for ratio in ratios_ecm_frc:
        ecm, frc = ratio
        ratio_name = f"{ecm}_{frc}"
        save_path = f"{output_dir}/collagenase_ratio_{ratio_name}.json"
        
        if os.path.exists(save_path):
            print(f"Ratio {ecm}:{frc} already exists, loading...")
            with open(save_path, 'r') as f:
                all_results[ratio_name] = json.load(f)
            continue
        
        # Running experiment with plotting
        results = run_collagenase_experiment(
            ratio_ecm_frc=ratio,
            num_tissues=num_tissues,
            ablations_per_tissue=ablations_per_tissue,
            total_le=total_le,
            save_path=save_path,
            plot_dir=plot_dir
        )
        
        all_results[ratio_name] = results
    
    # Saving master summary
    master_path = f"{output_dir}/all_ratios_summary.json"
    with open(master_path, 'w') as f:
        serializable_results = {}
        for name, data in all_results.items():
            serializable_results[name] = {
                'ratio': data['ratio'],
                'ecm_ratio': data['ecm_ratio'],
                'frc_ratio': data['frc_ratio'],
                'summary': data['summary']
            }
        json.dump(serializable_results, f, indent=2)
    
    print(f"All experiments complete!")
    print(f"Results saved to: {output_dir}")
    print(f"Summary: {master_path}")
    if plot_dir:
        print(f"Plots saved to: {plot_dir}")
    
    return all_results

In [ ]:
# Defining ratios as (ECM, FRC) tuples to be tested
ratios = [
    (1, 1), (1, 2), (1, 3), (1, 4), (1, 5),  # FRC dominant
    (2, 1), (3, 1), (4, 1), (5, 1),          # ECM dominant
    (6, 1), (7, 1), (8, 1), (9, 1), (10, 1)
]

# ratios = [(6,1)] uncomment this to test a single ratio

# Running experiment 
results = run_multi_ratio_collagenase_experiment(
    ratios_ecm_frc=ratios,
    num_tissues=3,           # 3 independently initialised tissues
    ablations_per_tissue=5,  # 5 consecutive ablations per tissue
    total_le=90,             # Total length elasticity to be split between FRC and ECM
    output_dir="Model1_collagenase_results",
    plot_dir="Model1_collagenase_plots"
)

### Model 2 collagenase laser ablations

In [ ]:
def initialise_tissue_model2_with_collagenase():
    """
    Initialising Model 2
    """
    # Initialising tissue
    cellmap, geom, energy_model = vertexModel2.initialize(40)
    cellmap = MechanicalParams2.update(cellmap)
    
    
    # Fixing outer boundary vertices with high viscosity
    boundary_edges, boundary_faces, inside_edges, outside_edges, inside_faces, inside_vertices, outside_vertices = auxFunctions.identify_boundary_layers(cellmap, 1)
    
    for vertex_id in outside_vertices:
        if vertex_id in cellmap.vert_df.index:
            cellmap.vert_df.at[vertex_id, "viscosity"] = 1000000
                    

    cellmap.update_specs({"vert": {"viscosity": cellmap.vert_df["viscosity"].values}}, reset=True)

    # Allowing tissue to relax after initialisation
    energy_model.compute_energy(cellmap)
    cellmap, geom, energy_model, _, _ = vertexModel2.solveEuler(
        cellmap, geom, energy_model, endTime=100
    )
    
    # Setting length_elasticity = 0 to simulate complete ECM degradation
    cellmap.edge_df["length_elasticity"] = 0.0
    
    # Allowinf tissue to relax after losing ECM
    energy_model.compute_energy(cellmap)
    cellmap, geom, energy_model, _, _ = vertexModel2.solveEuler(
        cellmap, geom, energy_model, endTime=100
    )
    
    return cellmap, geom, energy_model

def perform_laser_ablation_model2(cellmap, geom, energy_model, endTime=50, tissue_id=None, ablation_id=None, plot_dir=None):
    """
    Perform one laser ablation with visualisation.
    """
    # Finding interior edges
    boundary_edges, boundary_faces, inside_edges, outside_edges, inside_faces, inside_vertices, outside_vertices = auxFunctions.identify_boundary_layers(cellmap, max_layers=1)
    
    # Choosing random edge and finding its opposite edge
    chosen_edge = random.choice(inside_edges)
    srce = cellmap.edge_df.loc[chosen_edge, "srce"]
    trgt = cellmap.edge_df.loc[chosen_edge, "trgt"]
    
    opposite = cellmap.edge_df[
        (cellmap.edge_df["srce"] == trgt) & (cellmap.edge_df["trgt"] == srce)
    ]
    if opposite.empty:
        raise ValueError("Opposite edge not found")
    opposite_edge = opposite.index[0]
    
    # Visualising edge choice before ablation
    if plot_dir and ablation_id is not None:
        print(f"Tissue before ablation")
        auxFunctions.highlight_edge_on_cellmap(
            cellmap, 
            edge_id=chosen_edge,
            figsize=(12, 12),
            vert_col='orange',
            edge_highlight_col='red',
            save_path=f"{plot_dir}/before_ablation_tissue{tissue_id}_abl{ablation_id}.png",
            show_figure=False
        )
    
    # Ablating chosen edge
    cellmap.edge_df.at[chosen_edge, "length_elasticity"] = 0.0
    cellmap.edge_df.at[chosen_edge, "line_tension"] = 0.0
    cellmap.edge_df.at[opposite_edge, "length_elasticity"] = 0.0
    cellmap.edge_df.at[opposite_edge, "line_tension"] = 0.0
    
    # Relaxing the model
    energy_model.compute_energy(cellmap)
    cellmap, geom, energy_model, _, solver = vertexModel2.solveEuler(
        cellmap, geom, energy_model, endTime=endTime
    )
    
    # Visualising after ablation
    if plot_dir and ablation_id is not None:
        print(f"Tissue after ablation")
        auxFunctions.highlight_edge_on_cellmap(
            cellmap, 
            edge_id=chosen_edge,
            figsize=(12, 12),
            vert_col='orange',
            edge_highlight_col='red',
            save_path=f"{plot_dir}/after_ablation_tissue{tissue_id}_abl{ablation_id}.png",
            show_figure=False
        )
    
    # Tracking vertex displacement post ablation
    displacement = []
    time_steps = []
    initial_distance = None
    
    for t, cm in solver.history:
        srce_coords = cm.vert_df.loc[srce, ['x', 'y']]
        trgt_coords = cm.vert_df.loc[trgt, ['x', 'y']]
        distance = np.linalg.norm(srce_coords - trgt_coords)
        
        if initial_distance is None:
            initial_distance = distance
        
        displacement.append(distance - initial_distance)
        time_steps.append(t)
    
    # Fitting exponential recoil model (same fitting as carried out for ex vivo data)
    def recoil_model(t, initial_recoil, k):
        return (initial_recoil / k) * (1 - np.exp(-k * t))
    
    params, _ = curve_fit(recoil_model, time_steps, displacement, 
                          p0=[0.00001, 3], bounds=(0, np.inf))
    initial_recoil, k = params
    
    # Plotting recoil
    if plot_dir and ablation_id is not None:
        plot_recoil_curve(
            time_steps, displacement, initial_recoil, k, 
            tissue_id, ablation_id
        )
    
    return {
        'displacement': displacement,
        'time_steps': time_steps,
        'initial_recoil': float(initial_recoil),
        'k': float(k),
        'chosen_edge': int(chosen_edge),
        'opposite_edge': int(opposite_edge),
        'updated_cellmap': cellmap
    }


def run_model2_experiment(num_tissues=3, 
                          ablations_per_tissue=5,
                          save_path=None,
                          plot_dir=None):
    """
    Run Model 2 experiment across multiple independently-generated tissues,
    performing several consecutive ablations on each one
    """
    print("Model 2: Collagenase-treated tissue (no ECM)")

    if plot_dir:
        Path(plot_dir).mkdir(parents=True, exist_ok=True)
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    
    results = {
        'model': 'Model 2',
        'description': 'Collagenase-treated tissue (no ECM), fixed boundaries',
        'num_tissues': num_tissues,
        'ablations_per_tissue': ablations_per_tissue,
        'ablations': []
    }    
    
    for tissue_idx in range(num_tissues):
        print(f"Tissue {tissue_idx + 1}/{num_tissues}")
        
        # Initialising tissue
        print("Initialising collagenase-treated tissue")
        cellmap, geom, energy_model = initialise_tissue_model2_with_collagenase()
          
        # Performing consecutive laser ablations
        for ablation_idx in range(ablations_per_tissue):
            ablation_num = tissue_idx * ablations_per_tissue + ablation_idx + 1
            print(f"\nAblation {ablation_idx + 1}/{ablations_per_tissue} (Total: {ablation_num})")
            
            try:
                ablation_result = perform_laser_ablation_model2(
                    cellmap, geom, energy_model, endTime=50,
                    tissue_id=tissue_idx + 1,
                    ablation_id=ablation_num,
                    plot_dir=plot_dir
                )
                
                cellmap = ablation_result['updated_cellmap']  
                del ablation_result['updated_cellmap']

                
                ablation_result['tissue_id'] = tissue_idx + 1
                ablation_result['ablation_id'] = ablation_num
                
                results['ablations'].append(ablation_result)
                
                print(f"Ablation {ablation_num} complete!")
                print(f"K = {ablation_result['k']:.3f}")
                print(f"Initial recoil = {ablation_result['initial_recoil']:.5f}")
                
            except Exception as e:
                print(f"Ablation failed: {e}")
                traceback.print_exc()
                continue
    
    # Summary statistics
    if results['ablations']:
        k_values = [a['k'] for a in results['ablations']]
        recoil_values = [a['initial_recoil'] for a in results['ablations']]
        
        results['summary'] = {
            'num_ablations': len(results['ablations']),
            'mean_k': float(np.mean(k_values)),
            'std_k': float(np.std(k_values)),
            'mean_initial_recoil': float(np.mean(recoil_values)),
            'std_initial_recoil': float(np.std(recoil_values))
        }
    else:
        results['summary'] = {
            'num_ablations': 0,
            'mean_k': None,
            'std_k': None,
            'mean_initial_recoil': None,
            'std_initial_recoil': None
        }
    
    print(f"Model 2 Complete!")
    print(f"Summary Statistics:")
    print(f"   Successful ablations: {len(results['ablations'])}/{num_tissues * ablations_per_tissue}")
    if results['summary']['mean_k']:
        print(f"   Mean K: {results['summary']['mean_k']:.3f} ± {results['summary']['std_k']:.3f}")
        print(f"   Mean Initial Recoil: {results['summary']['mean_initial_recoil']:.5f} ± {results['summary']['std_initial_recoil']:.5f}")
    
    if save_path:
        with open(save_path, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"\nResults saved to: {save_path}")
    
    
    return results


In [ ]:
results = run_model2_experiment(
    num_tissues=3,
    ablations_per_tissue=5,
    save_path="Model2_collagenase_results/model2_summary.json",
    plot_dir="Model2_collagenase_plots"
)

In [ ]:
import json
import matplotlib.pyplot as plt

# Model 1 — only the 5:1 ECM:FRC ratio
with open("Model1_collagenase_results/collagenase_ratio_6_1.json") as f:
    model1_data = json.load(f)

model1_k = [a['k'] for a in model1_data['ablations']]
model1_recoil = [a['initial_recoil'] for a in model1_data['ablations']]

# Model 2 — no ratio, just the one summary file
with open("Model2_collagenase_results/model2_summary.json") as f:
    model2_data = json.load(f)

model2_k = [a['k'] for a in model2_data['ablations']]
model2_recoil = [a['initial_recoil'] for a in model2_data['ablations']]

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(model1_k, model1_recoil, label='Model 1 (5:1 ECM:FRC)', color='tab:blue')
ax.scatter(model2_k, model2_recoil, label='Model 2 (collagenase, no ECM)', color='tab:orange')

ax.set_xlabel('K (elasticity : viscosity ratio)')
ax.set_ylabel('Initial recoil')
ax.set_xlim(0, 0.25)
ax.set_ylim(0, 1)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()